In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/bench/checkpoints/pre_cell_6.pickle

In [4]:
%%RecordEvent
%%time
### cell 6 ###

### cell 6 optimized ###

# 0) Define the subset of countries used for bucketing
subset_of_countries = [
    'Other', 'India', 'United States of America', 'Brazil', 'Nigeria',
    'Pakistan', 'Japan', 'China', 'Egypt', 'Indonesia', 'Mexico',
    'Turkey', 'Russia'
]

# 1) Define the question and x-axis title
question_of_interest_cell18 = 'In which country do you currently reside?'
title_for_x_axis = ''

# 2) Optimize the count → percentage function
#    (preserve original denominator semantics: non-null count)
def count_then_return_percent_18(dataframe, x_axis_title):
    return (
        dataframe[x_axis_title]
        .value_counts(dropna=False)
        .mul(100)
        .div(dataframe[x_axis_title].count())
        .round(1)
    )

# 3a) Fix 2022 UK naming in place (values only)
responses_df_2022_cell10.replace(
    'United Kingdom (UK)',
    'United Kingdom of Great Britain and Northern Ireland',
    inplace=True
)

# 3b) Rename 2017 country column and remap values
responses_df_2017.rename(
    columns={
        'Country': question_of_interest_cell18,
        'CurrentJobTitleSelect':
            'Select the title most similar to your current role (or most recent title if retired): - Selected Choice'
    },
    inplace=True
)
responses_df_2017[question_of_interest_cell18].replace(
    {
        'United States': 'United States of America',
        "People 's Republic of China": 'China',
        'United Kingdom': 'United Kingdom of Great Britain and Northern Ireland'
    },
    inplace=True
)

# 4) Bucket any non-subset country value into 'Other'
#    (use the same chained‐indexing pattern as original code to match side‐effects)
for df in [
    responses_df_2017,
    responses_df_2018_cell10,
    responses_df_2019_cell10,
    responses_df_2020,
    responses_df_2021,
    responses_df_2022_cell10
]:
    df[question_of_interest_cell18][
        ~df[question_of_interest_cell18].isin(subset_of_countries)
    ] = 'Other'

# 5) Combine data from multiple years into one DataFrame
def combine_subset_of_data_from_multiple_years_18(
    question_of_interest,
    x_axis_title,
    include_2017=False
):
    year_to_df = {
        '2018': responses_df_2018_cell10,
        '2019': responses_df_2019_cell10,
        '2020': responses_df_2020,
        '2021': responses_df_2021,
        '2022': responses_df_2022_cell10
    }
    years = ['2018', '2019', '2020', '2021', '2022']
    if include_2017:
        year_to_df['2017'] = responses_df_2017
        years.insert(0, '2017')

    dfs = []
    axis_col = x_axis_title if x_axis_title else question_of_interest
    for year in years:
        pct = (
            count_then_return_percent_18(
                year_to_df[year], question_of_interest
            )
            .sort_index()
        )
        temp = pct.reset_index()
        temp.columns = [axis_col, 'percentage']
        temp['year'] = year
        dfs.append(temp)

    df_combined = pd.concat(dfs, ignore_index=True)
    return df_combined[['percentage', 'year', axis_col]]

# 6) Execute the combine and final cleanups
a_country_df = combine_subset_of_data_from_multiple_years_18(
    question_of_interest_cell18,
    title_for_x_axis,
    include_2017=True
)
country_df_combined_cell18 = (
    a_country_df
    .sort_values(by=['year', 'percentage'], ascending=True)
)
# Collapse the long UK name back to the shorter form
country_df_combined_cell18.replace(
    'United Kingdom of Great Britain and Northern Ireland',
    'United Kingdom',
    inplace=True
)

# 7) Display info to confirm success
country_df_combined_cell18.info()

<class 'cudf.core.dataframe.DataFrame'>
Index: 309 entries, 2 to 271
Data columns (total 3 columns):
 #   Column                                     Non-Null Count  Dtype
---  ------                                     --------------  -----
 0   percentage                                 309 non-null    float64
 1   year                                       309 non-null    object
 2   In which country do you currently reside?  309 non-null    object
dtypes: float64(1), object(2)
memory usage: 11.1+ KB
CPU times: user 1.12 s, sys: 15.4 ms, total: 1.14 s
Wall time: 1.13 s


In [ ]:
%Checkpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small/checkpoints/post_cell_6_try_4.pickle